In [1]:
import pandas as pd
import numpy as np
import json
import requests
from datetime import datetime, timedelta
from functions import prodFetch, trinoFetch, read, to_bigquery, bqFetch

# run mode
MANUAL_LOOKBACK_DAYS = None  # set to int when running manually

if MANUAL_LOOKBACK_DAYS:
    RUN_MODE = "manual"
    lookback = MANUAL_LOOKBACK_DAYS
else:
    RUN_MODE = "cron"
    lookback = 7
    
print(f"Run mode: {RUN_MODE} | Lookback: {lookback} days")

BQ_TABLE     = "yashesh_prod.mosfet_tracking"
BQ_PROJECT   = "bs-analytics-yashesh"
UPSERT_KEYS  = ["batteryId", "serialNo", "action_timestamp"]

Run mode: cron | Lookback: 7 days


In [2]:
# =========================================
# STEP 1: FETCH BQ STATE & LOGS
# =========================================

bq_existing = bqFetch(f"""
SELECT
    batteryId,
    serialNo,
    action_timestamp,
    last_updated
FROM `{BQ_PROJECT}.{BQ_TABLE}`
""")

bq_existing["action_timestamp"] = pd.to_datetime(bq_existing["action_timestamp"])
bq_existing["last_updated"]     = pd.to_datetime(bq_existing["last_updated"])

# rows with open window
bq_reprocess = bq_existing[
    bq_existing["action_timestamp"] + pd.Timedelta(days=7) > bq_existing["last_updated"]
][["batteryId", "serialNo", "action_timestamp"]].copy()

print(f"BQ rows with open window: {len(bq_reprocess)}")

2026-07-07 17:56:06,599 - INFO - Executing query on BigQuery...


[WARN] Using fallback credentials path
Downloading: 100%|██████████|

2026-07-07 17:56:11,837 - INFO - BigQuery fetch successful. Rows returned: 9988



BQ rows with open window: 2571


In [3]:
bq_reprocess['action_timestamp'].min()

Timestamp('2026-06-30 16:10:04')

In [4]:
# =========================================
# STEP 2: FETCH LOGS & IDENTIFY NEW ROWS
# =========================================

all_battery = read(
    "https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?gid=793657175#gid=793657175",
    "logs"
)

only_off = all_battery[all_battery["action"] == "turned_off"].copy()
only_off=only_off[only_off['action_date']>='2026-05-25']
# only_off["action_timestamp"] = pd.to_datetime(
#     pd.to_datetime(only_off["action_timestamp"]).dt.tz_localize(None)
# )
only_off["action_timestamp"] = pd.to_datetime(
    pd.to_datetime(only_off["action_timestamp"]).dt.tz_localize(None)
).dt.floor("s")
only_off['deviceId']=only_off['deviceId'].astype(str)
only_off['serialNo']=only_off['serialNo'].astype(str)
only_off['batteryId']=only_off['batteryId'].astype(str)
only_off['manufacturerName']=only_off['manufacturerName'].astype(str)
only_off = only_off.sort_values(
    ["batteryId", "action_timestamp"],
    ascending=[True, False]
).reset_index(drop=True)

# mosfet_turned_off derivation
def get_mosfet_turned_off(row):
    chg         = row["CHG CMD"]
    dischg      = row["DICHG CMD"]
    action_date = pd.to_datetime(row["action_date"]).date()
    cutoff      = pd.to_datetime("2026-06-09").date()
    chg_valid   = pd.notna(chg) and str(chg).strip() != ""
    dischg_valid= pd.notna(dischg) and str(dischg).strip() != ""

    if action_date <= cutoff:
        return "discharging"
    elif chg_valid and dischg_valid:
        return "both"
    elif chg_valid:
        return "charging"
    elif dischg_valid:
        return "discharging"
    else:
        return "unknown"

only_off["mosfet_turned_off"] = only_off.apply(get_mosfet_turned_off, axis=1)
only_off

2026-07-07 17:56:22,725 - INFO - Pygsheets client authorized successfully.


[AUTH] Using credentials: C:\Users\Yashesh.Pratap\Desktop\Repo\apackage\functions\analytics.json


,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,mosfet_turned_off
0,B1000134,60653FAB09468421,BATTRIXX,B112004420_131663,2026-07-04 16:10:46,2026-07-04,Discharging,turned_off,NaN,DISCHGOFF,NaN,6576361,26.875664,80.92292,discharging
1,B1000202,246F4B3F82B5A214,BATTRIXX,B112004420_131931,2026-06-02 16:16:41,2026-06-02,,turned_off,,,,,,,discharging
2,B1000852,80D52CC004987C7A,BATTRIXX,B112004450_127601,2026-06-24 00:08:37,2026-06-24,Charging,turned_off,CHGOFF,NaN,6526381,NaN,,,charging
3,B1000872,20327703C839920D,BATTRIXX,B112004450_127580,2026-06-24 00:07:50,2026-06-24,Charging,turned_off,NaN,DISCHGOFF,NaN,6433401,,,discharging
4,B1000880,B611B5624C10299F,BATTRIXX,B112004450_127387,2026-06-25 18:12:12,2026-06-25,Discharging,turned_off,CHGOFF,NaN,6593371,NaN,26.926434,75.84658,charging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10095,B999505,461FCF3311E51E77,BATTRIXX,B112004450_127104,2026-06-09 08:17:01,2026-06-09,,turned_off,,,,,,,discharging
10096,B999505,461FCF3311E51E77,BATTRIXX,B112004450_127104,2026-06-08 12:18:27,2026-06-08,,turned_off,,,,,,,discharging
10097,B999505,461FCF3311E51E77,BATTRIXX,B112004450_127104,2026-06-07 12:22:54,2026-06-07,,turned_off,,,,,,,discharging
10098,B999505,461FCF3311E51E77,BATTRIXX,B112004450_127104,2026-06-01 16:11:21,2026-06-01,,turned_off,,,,,,,discharging


In [5]:
# identify new rows not in BQ
logs_with_bq = only_off.merge(
    bq_existing[["batteryId", "serialNo", "action_timestamp"]],
    on=["batteryId", "serialNo", "action_timestamp"],
    how="left",
    indicator=True
)

new_rows = logs_with_bq[
    logs_with_bq["_merge"] == "left_only"
].drop(columns=["_merge"]).copy()

print(f"New rows not in BQ: {len(new_rows)}")

# reprocess rows with open window
reprocess_rows = only_off.merge(
    bq_reprocess,
    on=["batteryId", "serialNo", "action_timestamp"],
    how="inner"
)

only_off_to_process = pd.concat(
    [new_rows, reprocess_rows],
    ignore_index=True
).drop_duplicates(
    subset=["batteryId", "serialNo", "action_timestamp"]
).reset_index(drop=True)

print(f"Total rows to process: {len(only_off_to_process)}")

if len(only_off_to_process) == 0:
    print("Nothing to process this run. Exiting.")
    exit()

only_off_to_process

New rows not in BQ: 112
Total rows to process: 2683


,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,mosfet_turned_off
0,B1000921,104643F6F62AA79A,BATTRIXX,B112004450_127457,2026-07-07 14:09:31,2026-07-07,Discharging,turned_off,NaN,DISCHGOFF,NaN,6667924,26.933413,75.807686,discharging
1,B1023924,114B693621D0A836,STEFEN ELECTRIC,I24-58092,2026-07-07 12:08:38,2026-07-07,Discharging,turned_off,NaN,DISCHGOFF,NaN,2859361,28.620745,77.36861,discharging
2,B1029908,A1A52F8EBB3B693E,INVERTED,INCBZF5145E1152,2026-07-07 12:09:39,2026-07-07,Discharging,turned_off,CHGOFF,NaN,6719233,NaN,26.927128,75.84343,charging
3,B1030931,7A4AF13114311D64,INVERTED,INCBZF5145E1902,2026-07-07 12:08:57,2026-07-07,Discharging,turned_off,CHGOFF,NaN,6807801,NaN,26.466854,80.37016,charging
4,B190187,A6ADA10D9FE491D4,INVERTED,INDMWB514001044,2026-07-07 14:08:57,2026-07-07,Discharging,turned_off,CHGOFF,NaN,1189102,NaN,28.634771,77.10183,charging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2678,B999406,311EBB6CEC1799B6,BATTRIXX,B112004450_127122,2026-07-06 12:08:45,2026-07-06,Discharging,turned_off,CHGOFF,NaN,6103821,NaN,28.611645,77.35123,charging
2679,B999481,5A7245230437508E,BATTRIXX,B112004450_127117,2026-07-06 14:14:28,2026-07-06,Discharging,turned_off,CHGOFF,NaN,6104603,NaN,26.961481,75.87063,charging
2680,B999481,5A7245230437508E,BATTRIXX,B112004450_127117,2026-07-05 12:14:52,2026-07-05,Charging,turned_off,CHGOFF,NaN,6104602,NaN,26.961481,75.87063,charging
2681,B999485,A6409BADE7C01DF8,BATTRIXX,B112004450_127171,2026-07-06 14:14:33,2026-07-06,Discharging,turned_off,NaN,DISCHGOFF,NaN,6173303,26.961517,75.8706,discharging


In [6]:
earliest_action = only_off_to_process["action_timestamp"].min()
events_lo       = earliest_action - pd.Timedelta(days=1)
print(f"Fetching events from: {events_lo.date()}")
events_lo= events_lo.strftime('%Y-%m-%d %H:%M:%S')  # format to date string
print(events_lo)

Fetching events from: 2026-06-29
2026-06-29 16:10:04


## rolling fetch

In [7]:
# =========================================
# STEP 4: FETCH ALL EVENTS
# =========================================

all_txn = prodFetch(f"""
SELECT
    t.id AS receivingId,
    t.driverId,
    t.partnerId,
    t.createdAt + INTERVAL 330 MINUTE AS receiving_time,
    t.batteriesIssued,
    t.batteriesReceived,
    tmd.mappedFaceId
FROM transactions t
JOIN transactionsMetaDatas tmd ON t.id = tmd.transactionId
WHERE t.deletedAt IS NULL
AND t.createdAt >= '{events_lo}'
""")

all_inactivity = prodFetch(f"""
SELECT
    id AS receivingId,
    driverId,
    changedBy AS partnerId,
    batteriesReturned AS batteriesReceived,
    mappedFaceId,
    createdAt + INTERVAL 330 MINUTE AS receiving_time
FROM driverAudits
WHERE newStatus = 'onLeave'
AND deletedAt IS NULL
AND createdAt >= '{events_lo}'
""")

reconDfRaw = prodFetch(f"""
SELECT
    rb.batteryId,
    rb.foundSerialNo AS serialNo,
    rb.createdAt + INTERVAL 330 MINUTE AS recon_ts
FROM reconBatteries rb
WHERE rb.reconDate >= '{events_lo}'
AND rb.deletedAt IS NULL
AND rb.reconStatus IN ('correct', 'extra')
""")

all_services = prodFetch(f"""
SELECT
    assetId AS batteryId,
    createdAt + INTERVAL 330 MINUTE AS service_time
FROM services
WHERE createdAt >= '{events_lo}'
AND assetId LIKE 'B%'
""")

all_chg_dischg = trinoFetch(f"""
SELECT
    deviceId,
    state,
    startLat,
    startLon,
    startTs,
    endTs,
    startSoc,
    endSoc
FROM iceberg.silver.charging_discharging_events
WHERE deviceId IS NOT NULL
AND state IN ('charging', 'discharging')
AND startTs >= TIMESTAMP '{events_lo}'
AND startTs <= CURRENT_TIMESTAMP
AND ABS(endSoc - startSoc) > 10
""")

battery_df = prodFetch("""
SELECT
    id AS batteryId,
    serialNo,
    iotDeviceNo AS deviceId
FROM batteries
WHERE isBaas = 0
""")

battery_meta = prodFetch("""
SELECT
    manufacturerName,
    phase,
    batteryType,
    id AS batteryId,
    iotDeviceNo AS deviceId,
    serialNo
FROM batteries
WHERE isBaas = 0
""")

print(f"txn: {len(all_txn)} | inactivity: {len(all_inactivity)} | recon: {len(reconDfRaw)} | services: {len(all_services)} | chg_dischg: {len(all_chg_dischg)}")

2026-07-07 17:57:09,383 - INFO - Executing query on MySQL DB: batterysmart_prod
2026-07-07 17:57:09,711 - INFO - package: mysql.connector.plugins
2026-07-07 17:57:09,713 - INFO - plugin_name: mysql_native_password
2026-07-07 17:57:09,760 - INFO - AUTHENTICATION_PLUGIN_CLASS: MySQLNativePasswordAuthPlugin
2026-07-07 17:58:40,622 - INFO - batterysmart_prod → fetched 504359 rows
2026-07-07 17:58:40,697 - INFO - Executing query on MySQL DB: batterysmart_prod
2026-07-07 17:58:46,117 - INFO - batterysmart_prod → fetched 2887 rows
2026-07-07 17:58:46,121 - INFO - Executing query on MySQL DB: batterysmart_prod
2026-07-07 17:58:52,214 - INFO - batterysmart_prod → fetched 130269 rows
2026-07-07 17:58:52,227 - INFO - Executing query on MySQL DB: batterysmart_prod
2026-07-07 17:58:54,390 - INFO - batterysmart_prod → fetched 9677 rows
2026-07-07 17:58:54,408 - INFO - Executing Trino query on iceberg...
2026-07-07 18:00:49,792 - INFO - Trino query executed successfully. Rows fetched: 2197689
2026-07

txn: 504359 | inactivity: 2887 | recon: 130269 | services: 9677 | chg_dischg: 2197689


In [8]:
# =========================================
# STEP 5: CLEAN all_chg_dischg
# =========================================

all_chg_dischg["startTs"] = (
    pd.to_datetime(all_chg_dischg["startTs"])
    .dt.tz_localize(None)
)
all_chg_dischg["endTs"] = (
    pd.to_datetime(all_chg_dischg["endTs"])
    .dt.tz_localize(None)
)

all_chg_dischg = all_chg_dischg[
    all_chg_dischg["startTs"] >= pd.Timestamp(events_lo)
].reset_index(drop=True)

all_chg_dischg['duration'] = (
    all_chg_dischg['endTs'] - all_chg_dischg['startTs']
).abs().dt.total_seconds().floordiv(60).astype(int)

all_chg_dischg=all_chg_dischg[all_chg_dischg['duration']>=15]
all_chg_dischg

,deviceId,state,startLat,startLon,startTs,endTs,startSoc,endSoc,duration
0,000005861919085382991,discharging,25.677837,85.203800,2026-07-06 05:45:00,2026-07-06 20:55:00,99.0,20.0,910
1,000005861919085382991,charging,25.679270,85.202540,2026-07-06 21:00:00,2026-07-06 22:25:00,22.0,51.0,85
2,000005861919085571494,charging,28.986540,77.694984,2026-07-06 00:05:00,2026-07-06 08:55:00,71.0,98.0,530
3,000005861919085571494,discharging,28.984991,77.694890,2026-07-06 09:10:00,2026-07-06 20:50:00,97.0,31.0,700
4,000005861919085571494,charging,28.986555,77.694920,2026-07-06 20:55:00,2026-07-06 23:55:00,31.0,76.0,180
...,...,...,...,...,...,...,...,...,...
2116997,FEE7F1F91FB37FEC,charging,26.234423,78.163950,2026-07-02 11:55:00,2026-07-02 14:30:00,78.0,100.0,155
2116998,FEE7F1F91FB37FEC,discharging,26.234959,78.165610,2026-07-02 14:35:00,2026-07-02 20:20:00,99.0,51.0,345
2116999,FEE7F1F91FB37FEC,charging,26.234522,78.163860,2026-07-02 20:35:00,2026-07-02 22:40:00,52.0,91.0,125
2117000,FEEA19DD169B9D12,discharging,26.918669,80.959170,2026-07-02 06:50:00,2026-07-02 17:20:00,97.0,20.0,630


In [9]:
# =========================================
# STEP 6: PREPARE TXN + INACTIVITY
# =========================================

def safe_parse(x):
    if isinstance(x, list): return x
    if pd.isna(x) or x is None: return []
    try: return json.loads(x)
    except: return []

all_txn["event_type"]      = "txn"
all_inactivity["event_type"] = "inactivity"
all_inactivity["batteriesIssued"] = np.nan

cols = [
    "receivingId", "driverId", "partnerId",
    "receiving_time", "batteriesIssued",
    "batteriesReceived", "mappedFaceId", "event_type"
]

combined = pd.concat(
    [all_txn[cols], all_inactivity[cols]],
    ignore_index=True
).sort_values("receiving_time").reset_index(drop=True)

combined["batteriesIssued"]   = combined["batteriesIssued"].apply(safe_parse)
combined["batteriesReceived"] = combined["batteriesReceived"].apply(safe_parse)

issued   = combined.copy()
issued["batteryId"] = issued["batteriesIssued"]
issued   = issued[issued["batteryId"].apply(len) > 0].explode("batteryId")

received = combined.copy()
received["batteryId"] = received["batteriesReceived"]
received = received[received["batteryId"].apply(len) > 0].explode("batteryId")

txn_battery = pd.concat(
    [issued, received], ignore_index=True
)[["receivingId", "receiving_time", "driverId", "partnerId",
   "batteryId", "event_type", "mappedFaceId"]]

txn_battery = txn_battery[txn_battery["batteryId"].notna()].reset_index(drop=True)

battery_df["batteryId"] = battery_df["batteryId"].astype(str)
battery_df["deviceId"]  = battery_df["deviceId"].astype(str)
battery_df["serialNo"]  = battery_df["serialNo"].astype(str)

txn_battery["batteryId"] = txn_battery["batteryId"].astype(str)
txn_battery = txn_battery.merge(
    battery_df[["batteryId", "serialNo"]],
    on="batteryId", how="left"
)
txn_battery["receiving_time"] = pd.to_datetime(txn_battery["receiving_time"])
txn_battery["serialNo"]       = txn_battery["serialNo"].astype(str)
txn_battery = txn_battery.sort_values("receiving_time").reset_index(drop=True)

In [10]:
# =========================================
# STEP 7: MERGE ALL EVENTS
# =========================================

off_sorted = only_off_to_process.sort_values("action_timestamp").reset_index(drop=True)
off_sorted["serialNo"] = off_sorted["serialNo"].astype(str)
off_sorted["deviceId"] = off_sorted["deviceId"].astype(str)

# txn
txn_sorted = txn_battery[txn_battery["serialNo"].notna()].sort_values("receiving_time")
first_txn = pd.merge_asof(
    off_sorted,
    txn_sorted[["serialNo","receiving_time","driverId","partnerId","receivingId","event_type","mappedFaceId"]],
    left_on="action_timestamp", right_on="receiving_time",
    by="serialNo", direction="forward", tolerance=timedelta(days=7)
)

# recon
reconDfRaw["recon_ts"] = pd.to_datetime(reconDfRaw["recon_ts"])
reconDfRaw["serialNo"] = reconDfRaw["serialNo"].astype(str)
recon_sorted = reconDfRaw[reconDfRaw["serialNo"].notna()].sort_values("recon_ts")[["serialNo","recon_ts"]]
first_recon = pd.merge_asof(
    off_sorted,
    recon_sorted,
    left_on="action_timestamp", right_on="recon_ts",
    by="serialNo", direction="forward", tolerance=timedelta(days=7)
)

# service
all_services["service_time"] = pd.to_datetime(all_services["service_time"])
all_services["batteryId"]    = all_services["batteryId"].astype(str)
service_sorted = all_services.sort_values("service_time")[["batteryId","service_time"]]
first_service = pd.merge_asof(
    off_sorted,
    service_sorted,
    left_on="action_timestamp", right_on="service_time",
    by="batteryId", direction="forward", tolerance=timedelta(days=7)
)

# charging
charging_sorted = (
    all_chg_dischg[all_chg_dischg["state"]=="charging"]
    .sort_values("startTs")[["deviceId","startTs"]]
)
charging_sorted["deviceId"] = charging_sorted["deviceId"].astype(str)
first_chg = pd.merge_asof(
    off_sorted,
    charging_sorted,
    left_on="action_timestamp", right_on="startTs",
    by="deviceId", direction="forward", tolerance=timedelta(days=7)
).rename(columns={"startTs":"chg_ts"})

# discharging
discharging_sorted = (
    all_chg_dischg[all_chg_dischg["state"]=="discharging"]
    .sort_values("startTs")[["deviceId","startTs"]]
)
discharging_sorted["deviceId"] = discharging_sorted["deviceId"].astype(str)
first_dischg = pd.merge_asof(
    off_sorted,
    discharging_sorted,
    left_on="action_timestamp", right_on="startTs",
    by="deviceId", direction="forward", tolerance=timedelta(days=7)
).rename(columns={"startTs":"dischg_ts"})

# combine
join_cols = ["batteryId", "deviceId", "serialNo", "action_timestamp"]
combined_events = (
    first_txn
    .merge(first_recon[join_cols+["recon_ts"]],    on=join_cols, how="left")
    .merge(first_service[join_cols+["service_time"]], on=join_cols, how="left")
    .merge(first_chg[join_cols+["chg_ts"]],        on=join_cols, how="left")
    .merge(first_dischg[join_cols+["dischg_ts"]],  on=join_cols, how="left")
)
combined_events

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,...,receiving_time,driverId,partnerId,receivingId,event_type,mappedFaceId,recon_ts,service_time,chg_ts,dischg_ts
0,B837423,02B125AAB8E5B2B4,INVERTED,INDMWG514000442,2026-06-30 16:10:04,2026-06-30,Charging,turned_off,CHGOFF,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT
1,B816313,081DB4B9037AC9DF,BATTRIXX,B112004450_121124,2026-06-30 16:10:05,2026-06-30,Discharging,turned_off,NaN,DISCHGOFF,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,2026-07-01 08:55:00,2026-07-02 12:15:00
2,B614160,0A1FE1F89F90D703,TRONTEK,TK-5145-16LX-C-017069,2026-06-30 16:10:06,2026-06-30,Discharging,turned_off,NaN,DISCHGOFF,...,2026-07-05 11:47:18,D207542,P0243,68104418.0,txn,a9da08e0-127b-4722-a082-c5dd3abe0ba3,NaT,NaT,2026-07-02 08:25:00,2026-07-02 16:00:00
3,B576156,12BB3A43BE864921,INVERTED,INDMXL5140E1488,2026-06-30 16:10:07,2026-06-30,Discharging,turned_off,NaN,DISCHGOFF,...,2026-07-06 20:52:31,D91570,P3398,68207558.0,txn,c5b3a613-6812-4306-84b3-37ef8c6f1d4a,NaT,NaT,NaT,NaT
4,B729563,1543CF15A9483E9D,STEFEN ELECTRIC,G23-20079,2026-06-30 16:10:08,2026-06-30,Discharging,turned_off,CHGOFF,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,2026-06-30 16:25:00,2026-07-01 19:40:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2678,B537004,ACEA95CB094D700C,TRONTEK,TK-5145-03IX-C-007498,2026-07-07 14:09:32,2026-07-07,Charging,turned_off,NaN,DISCHGOFF,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT
2679,B709201,3B389AC0EE3D8396,IPOWER,HHL48452507010578,2026-07-07 14:09:33,2026-07-07,Charging,turned_off,NaN,DISCHGOFF,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT
2680,B694428,B10278E0E6B677AC,BATTRIXX,B112004180_130911,2026-07-07 14:09:34,2026-07-07,Discharging,turned_off,NaN,DISCHGOFF,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT
2681,B568003,69C74AC7EEA8059B,STEFEN ELECTRIC,E23-12288,2026-07-07 14:09:35,2026-07-07,Discharging,turned_off,NaN,DISCHGOFF,...,NaT,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT


In [11]:
# =========================================
# STEP 8: FORBIDDEN TS + FLAGS
# =========================================

def get_forbidden_ts(row):
    turned_off = row["mosfet_turned_off"]
    chg, dischg = row["chg_ts"], row["dischg_ts"]
    if turned_off == "charging":
        return chg
    elif turned_off == "discharging":
        return dischg
    elif turned_off == "both":
        candidates = [x for x in [chg, dischg] if pd.notna(x)]
        return min(candidates) if candidates else pd.NaT
    return pd.NaT

combined_events["forbidden_ts"] = combined_events.apply(get_forbidden_ts, axis=1)

combined_events["txn_flag"] = (
    combined_events["receiving_time"].notna()
) & (
    combined_events["forbidden_ts"].isna() |
    (combined_events["receiving_time"] <= combined_events["forbidden_ts"])
)

recon_candidate = (
    combined_events["recon_ts"].notna()
) & (~combined_events["txn_flag"]) & (
    combined_events["forbidden_ts"].isna() |
    (combined_events["recon_ts"] <= combined_events["forbidden_ts"])
)

service_candidate = (
    combined_events["service_time"].notna()
) & (~combined_events["txn_flag"]) & (
    combined_events["forbidden_ts"].isna() |
    (combined_events["service_time"] <= combined_events["forbidden_ts"])
)

both_exist = recon_candidate & service_candidate

combined_events["recon_flag"] = (
    recon_candidate & (
        ~both_exist |
        (combined_events["recon_ts"] <= combined_events["service_time"])
    )
)

combined_events["service_flag"] = (
    service_candidate & (
        ~both_exist |
        (combined_events["service_time"] < combined_events["recon_ts"])
    )
)

combined_events["api_failure_flag"] = (
    combined_events["forbidden_ts"].notna()
) & (~combined_events["txn_flag"]) & (
    ~combined_events["recon_flag"]
) & (~combined_events["service_flag"])

combined_events

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,...,mappedFaceId,recon_ts,service_time,chg_ts,dischg_ts,forbidden_ts,txn_flag,recon_flag,service_flag,api_failure_flag
0,B837423,02B125AAB8E5B2B4,INVERTED,INDMWG514000442,2026-06-30 16:10:04,2026-06-30,Charging,turned_off,CHGOFF,NaN,...,NaN,NaT,NaT,NaT,NaT,NaT,False,False,False,False
1,B816313,081DB4B9037AC9DF,BATTRIXX,B112004450_121124,2026-06-30 16:10:05,2026-06-30,Discharging,turned_off,NaN,DISCHGOFF,...,NaN,NaT,NaT,2026-07-01 08:55:00,2026-07-02 12:15:00,2026-07-02 12:15:00,False,False,False,True
2,B614160,0A1FE1F89F90D703,TRONTEK,TK-5145-16LX-C-017069,2026-06-30 16:10:06,2026-06-30,Discharging,turned_off,NaN,DISCHGOFF,...,a9da08e0-127b-4722-a082-c5dd3abe0ba3,NaT,NaT,2026-07-02 08:25:00,2026-07-02 16:00:00,2026-07-02 16:00:00,False,False,False,True
3,B576156,12BB3A43BE864921,INVERTED,INDMXL5140E1488,2026-06-30 16:10:07,2026-06-30,Discharging,turned_off,NaN,DISCHGOFF,...,c5b3a613-6812-4306-84b3-37ef8c6f1d4a,NaT,NaT,NaT,NaT,NaT,True,False,False,False
4,B729563,1543CF15A9483E9D,STEFEN ELECTRIC,G23-20079,2026-06-30 16:10:08,2026-06-30,Discharging,turned_off,CHGOFF,NaN,...,NaN,NaT,NaT,2026-06-30 16:25:00,2026-07-01 19:40:00,2026-06-30 16:25:00,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2678,B537004,ACEA95CB094D700C,TRONTEK,TK-5145-03IX-C-007498,2026-07-07 14:09:32,2026-07-07,Charging,turned_off,NaN,DISCHGOFF,...,NaN,NaT,NaT,NaT,NaT,NaT,False,False,False,False
2679,B709201,3B389AC0EE3D8396,IPOWER,HHL48452507010578,2026-07-07 14:09:33,2026-07-07,Charging,turned_off,NaN,DISCHGOFF,...,NaN,NaT,NaT,NaT,NaT,NaT,False,False,False,False
2680,B694428,B10278E0E6B677AC,BATTRIXX,B112004180_130911,2026-07-07 14:09:34,2026-07-07,Discharging,turned_off,NaN,DISCHGOFF,...,NaN,NaT,NaT,NaT,NaT,NaT,False,False,False,False
2681,B568003,69C74AC7EEA8059B,STEFEN ELECTRIC,E23-12288,2026-07-07 14:09:35,2026-07-07,Discharging,turned_off,NaN,DISCHGOFF,...,NaN,NaT,NaT,NaT,NaT,NaT,False,False,False,False


In [12]:
# =========================================
# STEP 9: IOT LATEST EVENT + SINK FLAG
# =========================================

latest_events = trinoFetch("""
SELECT deviceID AS deviceId, ts
FROM iceberg.silver.iot_events_latest
""")
latest_events["deviceId"] = latest_events["deviceId"].astype(str)
latest_events["ts"] = pd.to_datetime(latest_events["ts"]).dt.tz_localize(None)

combined_events["deviceId"] = combined_events["deviceId"].astype(str)
combined_events = combined_events.merge(latest_events, on="deviceId", how="left")

no_flag = (
    ~combined_events["txn_flag"] &
    ~combined_events["recon_flag"] &
    ~combined_events["service_flag"] &
    ~combined_events["api_failure_flag"]
)

combined_events["hours_since_action"] = (
    combined_events["ts"] - combined_events["action_timestamp"]
).dt.total_seconds() / 3600

combined_events["sink_flag"] = (
    no_flag &
    (
        combined_events["ts"].isna() |
        (combined_events["hours_since_action"] <= 48)
    )
)

combined_events["no_event_ping_flag"] = (
    no_flag & ~combined_events["sink_flag"]
)

2026-07-07 18:02:18,460 - INFO - Executing Trino query on iceberg...
2026-07-07 18:02:39,316 - INFO - Trino query executed successfully. Rows fetched: 400192


In [13]:
# =========================================
# STEP 10: DAYS DIFF + BATTERY META
# =========================================

combined_events["txn_days"] = (
    (combined_events["receiving_time"] - combined_events["action_timestamp"])
    .dt.total_seconds() / 86400
).where(combined_events["txn_flag"]).round(2)

combined_events["service_days"] = (
    (combined_events["service_time"] - combined_events["action_timestamp"])
    .dt.total_seconds() / 86400
).where(combined_events["service_flag"]).round(2)

combined_events["recon_days"] = (
    (combined_events["recon_ts"] - combined_events["action_timestamp"])
    .dt.total_seconds() / 86400
).where(combined_events["recon_flag"]).round(2)

combined_events["chgDis_days"] = (
    (combined_events["forbidden_ts"] - combined_events["action_timestamp"])
    .dt.total_seconds() / 86400
).where(combined_events["api_failure_flag"]).round(2)

battery_meta["batteryId"] = battery_meta["batteryId"].astype(str)
battery_meta["deviceId"]  = battery_meta["deviceId"].astype(str)
battery_meta["serialNo"]  = battery_meta["serialNo"].astype(str)

combined_events["batteryId"] = combined_events["batteryId"].astype(str)
combined_events["deviceId"]  = combined_events["deviceId"].astype(str)
combined_events["serialNo"]  = combined_events["serialNo"].astype(str)

combined_events = combined_events.merge(
    battery_meta[["batteryId","deviceId","serialNo","phase","batteryType"]],
    on=["batteryId","deviceId","serialNo"], how="left"
)

In [14]:
# =========================================
# STEP 11: FINAL COLUMN SELECTION + DTYPES
# =========================================

final_cols = [
    "batteryId", "deviceId", "manufacturerName", "serialNo",
    "mosfet_turned_off", "action_timestamp", "action_date",
    "receivingId", "receiving_time", "driverId", "partnerId",
    "event_type", "mappedFaceId", "recon_ts", "service_time",
    "chg_ts", "dischg_ts", "forbidden_ts",
    "txn_flag", "recon_flag", "service_flag",
    "api_failure_flag", "no_event_ping_flag", "sink_flag",
    "ts", "txn_days", "service_days", "recon_days", "chgDis_days",
    "phase", "batteryType"
]

combined_events_final = combined_events[final_cols].copy()

# fix dtypes
datetime_cols = [
    "action_timestamp", "receiving_time", "recon_ts",
    "service_time", "chg_ts", "dischg_ts", "forbidden_ts", "ts"
]
for col in datetime_cols:
    combined_events_final[col] = pd.to_datetime(combined_events_final[col], errors="coerce")

bool_cols = [
    "txn_flag", "recon_flag", "service_flag",
    "api_failure_flag", "no_event_ping_flag", "sink_flag"
]
for col in bool_cols:
    combined_events_final[col] = combined_events_final[col].astype(bool)

str_cols = [
    "batteryId", "deviceId", "manufacturerName", "serialNo",
    "mosfet_turned_off", "action_date", "driverId", "partnerId",
    "event_type", "mappedFaceId", "batteryType"
]
for col in str_cols:
    combined_events_final[col] = combined_events_final[col].astype(str).replace("nan", None)

float_cols = ["receivingId", "txn_days", "service_days", "recon_days", "chgDis_days", "phase"]
for col in float_cols:
    combined_events_final[col] = pd.to_numeric(combined_events_final[col], errors="coerce")
    
combined_events_final

,batteryId,deviceId,manufacturerName,serialNo,mosfet_turned_off,action_timestamp,action_date,receivingId,receiving_time,driverId,...,api_failure_flag,no_event_ping_flag,sink_flag,ts,txn_days,service_days,recon_days,chgDis_days,phase,batteryType
0,B837423,02B125AAB8E5B2B4,INVERTED,INDMWG514000442,charging,2026-06-30 16:10:04,2026-06-30,NaN,NaT,None,...,False,True,False,2026-07-07 17:56:29.009,NaN,NaN,NaN,NaN,1.0,NMC
1,B816313,081DB4B9037AC9DF,BATTRIXX,B112004450_121124,discharging,2026-06-30 16:10:05,2026-06-30,NaN,NaT,None,...,True,False,False,2026-07-07 17:57:51.430,NaN,NaN,NaN,1.84,2.0,LFP
2,B614160,0A1FE1F89F90D703,TRONTEK,TK-5145-16LX-C-017069,discharging,2026-06-30 16:10:06,2026-06-30,68104418.0,2026-07-05 11:47:18,D207542,...,True,False,False,2026-07-07 17:57:45.776,NaN,NaN,NaN,1.99,2.0,LFP
3,B576156,12BB3A43BE864921,INVERTED,INDMXL5140E1488,discharging,2026-06-30 16:10:07,2026-06-30,68207558.0,2026-07-06 20:52:31,D91570,...,False,False,False,2026-07-07 17:52:06.785,6.2,NaN,NaN,NaN,2.0,NMC
4,B729563,1543CF15A9483E9D,STEFEN ELECTRIC,G23-20079,charging,2026-06-30 16:10:08,2026-06-30,NaN,NaT,None,...,True,False,False,2026-07-07 17:36:04.971,NaN,NaN,NaN,0.01,1.0,NMC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2678,B537004,ACEA95CB094D700C,TRONTEK,TK-5145-03IX-C-007498,discharging,2026-07-07 14:09:32,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:37:15.907,NaN,NaN,NaN,NaN,2.0,LFP
2679,B709201,3B389AC0EE3D8396,IPOWER,HHL48452507010578,discharging,2026-07-07 14:09:33,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:57:55.576,NaN,NaN,NaN,NaN,2.0,LFP
2680,B694428,B10278E0E6B677AC,BATTRIXX,B112004180_130911,discharging,2026-07-07 14:09:34,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:57:49.232,NaN,NaN,NaN,NaN,2.0,LFP
2681,B568003,69C74AC7EEA8059B,STEFEN ELECTRIC,E23-12288,discharging,2026-07-07 14:09:35,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:50:32.759,NaN,NaN,NaN,NaN,1.0,NMC


In [ ]:
# # =========================================
# # ONE TIME: RECREATE BQ TABLE WITH TRUNCATED TIMESTAMPS
# # =========================================

# # fetch existing BQ data
# bq_existing_full = bqFetch(f"""
# SELECT * REPLACE(TIMESTAMP_TRUNC(action_timestamp, SECOND) AS action_timestamp)
# FROM `{BQ_PROJECT}.{BQ_TABLE}`
# """)

# # truncate in pandas as well to be safe
# bq_existing_full["action_timestamp"] = pd.to_datetime(
#     bq_existing_full["action_timestamp"]
# ).dt.floor("s")

# # fix all dtypes same as before
# datetime_cols = [
#     "action_timestamp", "receiving_time", "recon_ts",
#     "service_time", "chg_ts", "dischg_ts", "forbidden_ts", "ts"
# ]
# for col in datetime_cols:
#     bq_existing_full[col] = pd.to_datetime(bq_existing_full[col], errors="coerce")

# bool_cols = [
#     "txn_flag", "recon_flag", "service_flag",
#     "api_failure_flag", "no_event_ping_flag", "sink_flag"
# ]
# for col in bool_cols:
#     bq_existing_full[col] = bq_existing_full[col].astype(bool)

# str_cols = [
#     "batteryId", "deviceId", "manufacturerName", "serialNo",
#     "mosfet_turned_off", "action_date", "driverId", "partnerId",
#     "event_type", "mappedFaceId", "batteryType"
# ]
# for col in str_cols:
#     bq_existing_full[col] = bq_existing_full[col].astype(str).replace("nan", None)

# float_cols = ["receivingId", "txn_days", "service_days", "recon_days", "chgDis_days", "phase"]
# for col in float_cols:
#     bq_existing_full[col] = pd.to_numeric(bq_existing_full[col], errors="coerce")

# # replace table with clean data
# to_bigquery(
#     bq_existing_full,
#     destination_table=BQ_TABLE,
#     project_id=BQ_PROJECT,
#     if_exists="replace"  # wipe & recreate with truncated timestamps
# )

# print(f"BQ table recreated with second-truncated timestamps: {len(bq_existing_full)} rows")

# temp=bqFetch("SELECT * FROM yashesh_prod.mosfet_tracking order by action_timestamp")

2026-07-07 17:40:40,930 - INFO - Executing query on BigQuery...


Downloading: 100%|██████████|

2026-07-07 17:40:49,061 - INFO - Total time taken 8.12 s.
Finished at 2026-07-07 17:40:49.
2026-07-07 17:40:49,062 - INFO - BigQuery fetch successful. Rows returned: 9988


2026-07-07 17:40:49,298 - INFO - Uploading 9988 rows to BigQuery: bs-analytics-yashesh.yashesh_prod.mosfet_tracking (mode: replace)...
9988 out of 9988 rows loaded., ?it/s]2026-07-07 17:40:54,982 - INFO - 
100%|██████████| 1/1 [00:00<00:00, 164.35it/s]
2026-07-07 17:40:54,993 - INFO - Successfully uploaded data to BigQuery table: yashesh_prod.mosfet_tracking.
2026-07-07 17:40:55,048 - INFO - Executing query on BigQuery...


BQ table recreated with second-truncated timestamps: 9988 rows
Downloading: 100%|██████████|

2026-07-07 17:41:01,402 - INFO - Total time taken 6.34 s.
Finished at 2026-07-07 17:41:01.
2026-07-07 17:41:01,405 - INFO - BigQuery fetch successful. Rows returned: 9988


In [15]:
combined_events_final["action_timestamp"] = pd.to_datetime(
    combined_events_final["action_timestamp"]
).dt.floor("s")
combined_events_final

,batteryId,deviceId,manufacturerName,serialNo,mosfet_turned_off,action_timestamp,action_date,receivingId,receiving_time,driverId,...,api_failure_flag,no_event_ping_flag,sink_flag,ts,txn_days,service_days,recon_days,chgDis_days,phase,batteryType
0,B837423,02B125AAB8E5B2B4,INVERTED,INDMWG514000442,charging,2026-06-30 16:10:04,2026-06-30,NaN,NaT,None,...,False,True,False,2026-07-07 17:56:29.009,NaN,NaN,NaN,NaN,1.0,NMC
1,B816313,081DB4B9037AC9DF,BATTRIXX,B112004450_121124,discharging,2026-06-30 16:10:05,2026-06-30,NaN,NaT,None,...,True,False,False,2026-07-07 17:57:51.430,NaN,NaN,NaN,1.84,2.0,LFP
2,B614160,0A1FE1F89F90D703,TRONTEK,TK-5145-16LX-C-017069,discharging,2026-06-30 16:10:06,2026-06-30,68104418.0,2026-07-05 11:47:18,D207542,...,True,False,False,2026-07-07 17:57:45.776,NaN,NaN,NaN,1.99,2.0,LFP
3,B576156,12BB3A43BE864921,INVERTED,INDMXL5140E1488,discharging,2026-06-30 16:10:07,2026-06-30,68207558.0,2026-07-06 20:52:31,D91570,...,False,False,False,2026-07-07 17:52:06.785,6.2,NaN,NaN,NaN,2.0,NMC
4,B729563,1543CF15A9483E9D,STEFEN ELECTRIC,G23-20079,charging,2026-06-30 16:10:08,2026-06-30,NaN,NaT,None,...,True,False,False,2026-07-07 17:36:04.971,NaN,NaN,NaN,0.01,1.0,NMC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2678,B537004,ACEA95CB094D700C,TRONTEK,TK-5145-03IX-C-007498,discharging,2026-07-07 14:09:32,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:37:15.907,NaN,NaN,NaN,NaN,2.0,LFP
2679,B709201,3B389AC0EE3D8396,IPOWER,HHL48452507010578,discharging,2026-07-07 14:09:33,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:57:55.576,NaN,NaN,NaN,NaN,2.0,LFP
2680,B694428,B10278E0E6B677AC,BATTRIXX,B112004180_130911,discharging,2026-07-07 14:09:34,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:57:49.232,NaN,NaN,NaN,NaN,2.0,LFP
2681,B568003,69C74AC7EEA8059B,STEFEN ELECTRIC,E23-12288,discharging,2026-07-07 14:09:35,2026-07-07,NaN,NaT,None,...,False,False,True,2026-07-07 17:50:32.759,NaN,NaN,NaN,NaN,1.0,NMC


In [16]:
# =========================================
# SANITY CHECK - CURRENT FLAG DISTRIBUTION
# (before second truncation changes)
# =========================================

flag_cols = [
    "txn_flag", "recon_flag", "service_flag",
    "api_failure_flag", "no_event_ping_flag", "sink_flag"
]

print("Current flag distribution:")
for f in flag_cols:
    print(f"  {f}: {combined_events_final[f].sum()}")

print(f"\nTotal rows: {len(combined_events_final)}")
print(f"\nFlag count distribution (should all be 1):")
print(combined_events_final[flag_cols].sum(axis=1).value_counts())

Current flag distribution:
  txn_flag: 420
  recon_flag: 20
  service_flag: 4
  api_failure_flag: 1441
  no_event_ping_flag: 303
  sink_flag: 495

Total rows: 2683

Flag count distribution (should all be 1):
1    2683
Name: count, dtype: int64


In [17]:
# =========================================
# STEP 12: STAMP last_updated & UPSERT TO BQ
# =========================================

combined_events_final["last_updated"] = pd.Timestamp.now()

to_bigquery(
    combined_events_final,
    destination_table=BQ_TABLE,
    project_id=BQ_PROJECT,
    unique_keys=UPSERT_KEYS
)

print(f"Successfully upserted {len(combined_events_final)} rows to BQ")

# sanity
flag_cols = [
    "txn_flag", "recon_flag", "service_flag",
    "api_failure_flag", "no_event_ping_flag", "sink_flag"
]
print("\nFlag distribution:")
for f in flag_cols:
    print(f"  {f}: {combined_events_final[f].sum()}")

2026-07-07 18:06:40,304 - INFO - Upsert triggered. Uploading staging data to yashesh_prod.mosfet_tracking_staging...
2683 out of 2683 rows loaded., ?it/s]2026-07-07 18:06:49,622 - INFO - 
100%|██████████| 1/1 [00:00<00:00, 287.22it/s]
2026-07-07 18:06:49,632 - INFO - Executing MERGE onto destination table: yashesh_prod.mosfet_tracking...
2026-07-07 18:06:56,941 - INFO - Cleaning up staging table yashesh_prod.mosfet_tracking_staging...
2026-07-07 18:06:57,111 - INFO - Successfully upserted data to BigQuery table: yashesh_prod.mosfet_tracking.


Successfully upserted 2683 rows to BQ

Flag distribution:
  txn_flag: 420
  recon_flag: 20
  service_flag: 4
  api_failure_flag: 1441
  no_event_ping_flag: 303
  sink_flag: 495
